In [16]:

!pip install category_encoders
!pip install catboost
import json
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import category_encoders as ce
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score
from pathlib import Path
from datetime import datetime

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [17]:
# Load TRAINING data (not annotated - we create pseudo-labels)
with open("../../files/linkedin-cvs-not-annotated.json", "r", encoding="utf-8") as f:
    cvs_train = json.load(f)

# Load EVALUATION data (annotated - ground truth)
with open("../../files/linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs_eval = json.load(f)

print(f"Training CVs: {len(cvs_train)}")
print(f"Evaluation CVs: {len(cvs_eval)}")

Training CVs: 390
Evaluation CVs: 609


In [20]:
from datetime import datetime
CURRENT_YEAR = datetime.now().year
def extract_year(date_str):
    if not date_str:
        return None
    try:
        return int(date_str[:4])
    except:
        return None

def extract_features_from_cvs(cvs):
    """Extract features from CV list"""
    jobs = []
    
    for person_id, cv in enumerate(cvs):
        # 1. find previous jobs
        num_prev_jobs = sum(
            1 for job_item in cv if job_item.get("status") != "ACTIVE"
        )

        # 2. find the individual's start working time
        start_years = []
        for job_item in cv:
            year = extract_year(job_item.get("startDate"))
            if year:
                start_years.append(year)

        first_year = min(start_years) if start_years else None

        # 3. extract ACTIVE jobs
        for job in cv:
            if job.get("status") == "ACTIVE":
                total_years_experience = None
                if first_year:
                    total_years_experience = CURRENT_YEAR - first_year

                jobs.append({
                    **job,
                    "person_id": person_id,
                    "num_previous_jobs": num_prev_jobs,
                    "total_years_experience": total_years_experience
                })
    
    return pd.DataFrame(jobs)

# Extract features for TRAINING data (not annotated)
df_train = extract_features_from_cvs(cvs_train)

# Extract features for EVALUATION data (annotated)
df_eval = extract_features_from_cvs(cvs_eval)

print(f"\nTraining samples (ACTIVE jobs): {len(df_train)}")
print(f"Evaluation samples (ACTIVE jobs): {len(df_eval)}")


Training samples (ACTIVE jobs): 419
Evaluation samples (ACTIVE jobs): 623


In [22]:
# Load seniority definitions for rule-based matching
df_seniority = pd.read_csv("../../files/seniority-v2.csv")

seniority_dict = (
    df_seniority
    .groupby("label")["text"]
    .apply(list)
    .to_dict()
)

def predict_seniority_baseline(position, seniority_dict):
    """Rule-based seniority prediction for pseudo-labeling"""
    if pd.isna(position):
        return "Professional"  # Default
    
    position_lower = position.lower()
    
    for label, texts in seniority_dict.items():
        for text in texts:
            if text.lower() in position_lower:
                return label
    
    return "Professional"  # Default if no match

# Create PSEUDO-LABELS for training data
df_train["seniority"] = df_train["position"].apply(
    lambda x: predict_seniority_baseline(x, seniority_dict)
)

print("\nPseudo-label distribution (training):")
print(df_train["seniority"].value_counts())


Pseudo-label distribution (training):
seniority
Professional    163
Senior           89
Management       85
Lead             41
Director         30
Junior           11
Name: count, dtype: int64


In [23]:
# Encode seniority (ordinal)
seniority_order = [["Junior", "Professional", "Senior", "Lead", "Management", "Director"]]

encoder = OrdinalEncoder(categories=seniority_order)

# Encode TRAINING labels (pseudo-labels)
df_train["seniority_encoded"] = encoder.fit_transform(df_train[["seniority"]])

# Encode EVALUATION labels (ground truth)
df_eval["seniority_encoded"] = encoder.transform(df_eval[["seniority"]])

In [24]:
position_map = {
    "intern": 0,
    "trainee": 0,
    "junior": 1,
    "associate": 1,
    "analyst": 2,
    "senior": 3,
    "lead": 4,
    "manager": 4,
    "head": 5,
    "director": 6,
    "vp": 7,
    "ceo": 8
}

def map_position_level(position):
    if pd.isna(position):
        return 2  # Default
    position_lower = position.lower()
    return next(
        (v for k, v in position_map.items() if k in position_lower),
        2
    )

# Map for BOTH datasets
df_train["position_level"] = df_train["position"].apply(map_position_level)
df_eval["position_level"] = df_eval["position"].apply(map_position_level)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

# Define models
random_forest = RandomForestClassifier(n_estimators=200, random_state=0)
CatBoost = CatBoostClassifier(verbose=False, random_seed=0)
Logistic_Regression = LogisticRegression(max_iter=5000)

print("Models initialized!")

Models initialized!


In [34]:
# Prepare features (OHNE department!)
feature_cols = ["position", "num_previous_jobs", "total_years_experience"]

X_train = df_train[feature_cols].copy()
y_train = df_train["seniority_encoded"]

X_eval = df_eval[feature_cols].copy()
y_eval = df_eval["seniority_encoded"]

# Fill missing values
X_train["num_previous_jobs"] = X_train["num_previous_jobs"].fillna(0)
X_eval["num_previous_jobs"] = X_eval["num_previous_jobs"].fillna(0)

X_train["total_years_experience"] = X_train["total_years_experience"].fillna(
    X_train["total_years_experience"].median()
)
X_eval["total_years_experience"] = X_eval["total_years_experience"].fillna(
    X_eval["total_years_experience"].median()
)

# Preprocessor (nur position encoding)
categorical_cols = ['position']
categorical_transformer = ce.CountEncoder(cols=categorical_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ], 
    remainder='passthrough'
)

# Train and evaluate
print("\n=== Count Encoding (without department) ===")
for model in [random_forest, CatBoost, Logistic_Regression]:
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_eval)
    acc = accuracy_score(y_eval, y_pred)
    print(f"{type(model).__name__} accuracy: {acc:.4f}")


=== Count Encoding (without department) ===
RandomForestClassifier accuracy: 0.3355
CatBoostClassifier accuracy: 0.3579
LogisticRegression accuracy: 0.2970
